In [2]:
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import decoupler as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

import rpy2.robjects as robjects
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ROOT = Path("..").resolve()
RES, FIG = ROOT / "results", ROOT / "figures" / "main"
ENR = RES / "enrichment"
RAW = ENR / "raw_decoupler"
RAW.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
SEED = 2026
MINSIZE, MAXSIZE = 10, 500
TIMES = {"hallmark": 50_000, "reactome": 50_000, "go_bp": 20_000}

EXCLUDE_HALLMARK = {
    "ADIPOGENESIS", "XENOBIOTIC_METABOLISM", "BILE_ACID_METABOLISM",
    "FATTY_ACID_METABOLISM", "COAGULATION", "ESTROGEN_RESPONSE_EARLY",
    "ALLOGRAFT_REJECTION",
}

TISSUES = ["Heart_and_Aorta", "Kidney", "Liver", "Lung"]
CELLTYPES = ["endothelial cell", "macrophage"]
NODE_ORDER = [f"{t}__{c}" for t in TISSUES for c in CELLTYPES]
COLLECTIONS = {
    "hallmark": ("mh.all.v2026.1.Mm.symbols.gmt", "HALLMARK_"),
    "reactome": ("m2.cp.v2026.1.Mm.symbols.gmt", "REACTOME_"),
    "go_bp":    ("m5.go.bp.v2026.1.Mm.symbols.gmt", "GOBP_"),
}
print("decoupler", dc.__version__, "| nodes", len(NODE_ORDER))

C:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


decoupler 2.2.0 | nodes 8


In [3]:
with open(ROOT / "all_results.pkl", "rb") as fh:
    de_results = pickle.load(fh)

stat_all = pd.DataFrame({f"{t}__{c}": d.set_index("variable")["stat"]
                         for (t, c), d in de_results.items()}).T
stat = stat_all.fillna(0.0).loc[NODE_ORDER]        # 关键：保留全部 DE 基因，不裁到集合内
print("排序矩阵:", stat.shape)

排序矩阵: (8, 8102)


## 2. 三集合 GSEA（decoupler，BH-FDR）

耗时约 1h45m。**若 `raw_decoupler/*_gsea_long.tsv` 已存在会自动跳过**，
想重算就先把对应文件删掉。

In [4]:
for coll, (gmt_name, prefix) in COLLECTIONS.items():
    out = RAW / f"{coll}_gsea_long.tsv"
    if out.exists():
        print(f"[{coll}] {out.name} 已存在，跳过")
        continue
    times = TIMES[coll]
    net = dc.pp.read_gmt(path=str(ROOT / "data_interim" / "gmt" / gmt_name))
    net = net[net["source"].str.startswith(prefix)].copy()
    sz = net.groupby("source")["target"].apply(lambda s: len(set(s) & set(stat.columns)))
    keep_src = sz[(sz >= MINSIZE) & (sz <= MAXSIZE)].index
    net = net[net["source"].isin(keep_src)].copy()

    t0 = time.time()
    es, padj = dc.mt.gsea(data=stat, net=net, times=times, seed=SEED, verbose=False)
    es, padj = es.loc[NODE_ORDER], padj.loc[NODE_ORDER]      # padj 已是 BH-FDR

    long = pd.concat([es.stack().rename("NES"), padj.stack().rename("padj")],
                     axis=1).reset_index()
    long.columns = ["node", "pathway", "NES", "padj"]
    long["collection"] = coll
    long["size"] = long["pathway"].map(sz).astype(int)
    long["direction"] = np.where(long["NES"] > 0, "up_in_old", "down_in_old")
    long["significant"] = long["padj"] < ALPHA
    long = (long[["node", "pathway", "collection", "NES", "padj",
                  "significant", "direction", "size"]]
            .sort_values(["node", "padj"]).reset_index(drop=True))
    long.to_csv(out, sep="\t", index=False)
    print(f"[{coll}] sets={net['source'].nunique()} times={times} "
          f"{time.time()-t0:.0f}s 显著(padj<{ALPHA})={int(long.significant.sum())}")

[hallmark] hallmark_gsea_long.tsv 已存在，跳过
[reactome] reactome_gsea_long.tsv 已存在，跳过
[go_bp] go_bp_gsea_long.tsv 已存在，跳过


## 3. 交付物① `gsea_all_nodes.tsv` 与 ② `pathway_matrix.csv`

In [5]:
L = pd.concat([pd.read_csv(RAW / f"{c}_gsea_long.tsv", sep="\t")
               for c in ["hallmark", "reactome", "go_bp"]], ignore_index=True)
L["tissue"] = L["node"].str.split("__").str[0]
L["celltype"] = L["node"].str.split("__").str[1]
L["method"] = "GSEA"
L["pathway_short"] = (L["pathway"].str.replace("HALLMARK_", "", regex=False)
                                .str.replace("REACTOME_", "", regex=False)
                                .str.replace("GOBP_", "", regex=False))

# 剔除已判定为伪影/污染的 Hallmark 通路（只删 Hallmark，不动 Reactome / GO:BP）
_before = len(L)
L = L[~(L["collection"].eq("hallmark") & L["pathway_short"].isin(EXCLUDE_HALLMARK))].copy()
print(f"剔除 {len(EXCLUDE_HALLMARK)} 条 Hallmark 通路：{_before} -> {len(L)} 行")

cols = ["node", "pathway", "NES", "padj", "collection", "method",
        "direction", "significant", "size", "tissue", "celltype", "pathway_short"]
allnodes = L[cols].sort_values(["collection", "node", "padj"]).reset_index(drop=True)
allnodes.to_csv(ENR / "gsea_all_nodes.tsv", sep="\t", index=False)

pm = (L[["tissue", "celltype", "pathway", "NES", "padj", "collection",
         "significant", "direction", "size"]]
      .rename(columns={"NES": "score"})
      .sort_values(["collection", "tissue", "celltype", "padj"]).reset_index(drop=True))
pm.to_csv(ENR / "pathway_matrix.csv", index=False)

print("gsea_all_nodes.tsv", allnodes.shape, "| pathway_matrix.csv", pm.shape)

剔除 7 条 Hallmark 通路：36424 -> 36368 行


gsea_all_nodes.tsv (36368, 12) | pathway_matrix.csv (36368, 9)


## 4. 共享 / 部分 / 特异模块 + 去冗余

- 模块判定：通路在**同一方向**上活跃的组织数 —— ≥3 组织 = Shared、2 = Partial、1 = Specific
- 去冗余按集合分别处理（代表通路用于 §5 图 3(b)(d) 与 §7 的展示）：
  - **GO:BP**：改用 **topGO**。逐节点以 BH-FDR < 0.05 的 DE 基因做 GO:BP 富集（Fisher 精确检验），
    算法取 **elim**——自底向上遍历 GO-DAG，子项显著时把其注释的基因从祖先项中扣除后再检验，
    使「仅继承子项信号」的祖先项不再显著；跨节点取并集后映射回 MSigDB 名称，再与 GSEA 显著条目取交集。
    1386 条 GSEA 显著 GO:BP 通路 → **244 条 topGO 代表**。
  - **Reactome / Hallmark**：不属于 GO 本体（topGO 只适用于 GO），仍按基因集 **Jaccard > 0.85**
    合并冗余条目，合并组内保留基因集最小（同大小时 |max NES| 最大）者为代表：268 → **245**、33 → **33**。
- 完整逐通路结果保留在 `gsea_all_nodes.tsv` / `pathway_matrix.csv`；
  代表清单见 `redundancy_representatives.tsv`，逐条标注见 `redundancy_clusters.tsv`（`is_representative`）。


In [6]:
sig = L[L.significant].copy()
act = (sig.groupby(["collection", "pathway", "pathway_short", "direction"], observed=True)
          .agg(n_nodes=("node", "nunique"),
               n_tissues=("tissue", "nunique"),
               tissues=("tissue", lambda s: ";".join(sorted(set(s)))),
               mean_NES=("NES", "mean"),
               max_abs_NES=("NES", lambda s: float(np.abs(s).max())))
          .reset_index())
act["module"] = np.where(act.n_tissues >= 3, "Shared",
                 np.where(act.n_tissues == 2, "Partial", "Specific"))
act = act.sort_values(["module", "n_tissues", "max_abs_NES"],
                      ascending=[True, False, False]).reset_index(drop=True)
act.to_csv(ENR / "module_assignment.tsv", sep="\t", index=False)

print("显著「通路 x 方向」条目:", len(act))
print(act.groupby(["collection", "module"], observed=True).size().unstack(fill_value=0))

显著「通路 x 方向」条目: 1852
module      Partial  Shared  Specific
collection                           
go_bp           269     222       968
hallmark          7      15        20
reactome         76      39       236


In [7]:
import re
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# ---- 1. 自包含加载 topGO + org.Mm.eg.db ----
ro.r('''
    suppressPackageStartupMessages({
        library(topGO)
        library(org.Mm.eg.db)
    })
''')

# ---- 2. padj 矩阵 ----
padj_all = pd.DataFrame({
    f"{t}__{c}": d.set_index("variable")["adj_p_value"]
    for (t, c), d in de_results.items()
}).T.loc[NODE_ORDER]

# ---- 3. GO:BP -> 逐节点 topGO elim ----
elim_kept_ids = {}   # node -> set(GO ID)
elim_pv = {}         # node -> {GO ID: elim p}   （REVIGO 需要逐项分数）
for node in NODE_ORDER:
    importance = padj_all.loc[node].reindex(stat.columns).fillna(1.0)
    r_imp = ro.FloatVector(importance.values)
    r_imp.names = ro.StrVector(importance.index.tolist())
    ro.r.assign("geneList", r_imp)

    ro.r('''
        GOdata <- new("topGOdata",
                      ontology = "BP",
                      allGenes = geneList,
                      geneSelectionFun = function(x) x < 0.05,
                      annot = annFUN.org,
                      mapping = "org.Mm.eg.db",
                      ID = "symbol",
                      nodeSize = 5)
        resultElim <- runTest(GOdata, algorithm = "elim",
                               statistic = "fisher",
                               cutOff = 0.1)
        tab <- GenTable(GOdata, elimFisher = resultElim, topNodes = 3000)
    ''')
    with localconverter(ro.default_converter + pandas2ri.converter):
        tab_df = ro.conversion.rpy2py(ro.r['tab'])
    tab_df["elimFisher"] = tab_df["elimFisher"].astype(float)
    sig_tab = tab_df.loc[tab_df["elimFisher"] < ALPHA, ["GO.ID", "elimFisher"]]
    elim_kept_ids[node] = set(sig_tab["GO.ID"].astype(str))
    elim_pv[node] = dict(zip(sig_tab["GO.ID"].astype(str), sig_tab["elimFisher"]))

all_elim_ids = set().union(*elim_kept_ids.values())
print(f"\n[GO:BP] topGO elim 保留 GO ID（跨节点并集）: {len(all_elim_ids)}")

# ---- 4. GO ID -> MSigDB 名称映射（利用本地 GO.db，无需联网）----
ro.r('''
    suppressPackageStartupMessages(library(GO.db))
    terms <- Term(GOTERM)
    term_df <- data.frame(GOID=names(terms), TERM=as.character(terms), stringsAsFactors=FALSE)
    term_df$GOBP_NAME <- paste0("GOBP_", toupper(gsub("[^A-Za-z0-9]+", "_", term_df$TERM)))
    term_df$GOBP_NAME <- gsub("_$", "", term_df$GOBP_NAME)
''')
with localconverter(ro.default_converter + pandas2ri.converter):
    term_df = ro.conversion.rpy2py(ro.r['term_df'])
id2short = dict(zip(term_df["GOID"], term_df["GOBP_NAME"].str.replace("GOBP_", "", regex=False)))

# ---- 4b. 先限定到「GSEA 显著」的 GO:BP 通路，再做 REVIGO ----
# 图是按 GSEA NES 上色、按 GSEA FDR 打星的，因此只在 GSEA 显著的条目里选代表；
# 若先在全量 1060 上聚类再取交，整簇代表一旦不显著，该簇信号会整体丢失。
gsea_sig_short = set(L.loc[(L.collection == "go_bp") & L.significant, "pathway_short"])
keep_ids = sorted({g for g in all_elim_ids if id2short.get(g) in gsea_sig_short})
print(f"[GO:BP] topGO 保留 ∩ GSEA 显著: {len(keep_ids)}")

# ---- 4c. 人工剔除与「多组织衰老共同轨迹」无关的条目 ----
# 这些条目多为小样本、低覆盖的 GO 项，在 topGO elim 之后仅凭少量 DE 基因即达显著，
# 主题上属神经/感官发育、粒细胞趋化、骨骼肌肉/胚胎附属结构等，不进入本文叙事。
# 剔除发生在 REVIGO 之前：剩余条目重新聚类，被删项若原为簇代表，簇内次优者自动接替。
EXCLUDE_GO = {
    # 神经 / 感官 / 胶质
    "GO:0098883",  # synapse pruning
    "GO:0150078",  # positive regulation of neuroinflammatory response
    "GO:0010977",  # negative regulation of neuron projection development
    "GO:0019226",  # transmission of nerve impulse
    "GO:0021952",  # central nervous system projection neuron axonogenesis
    "GO:0001764",  # neuron migration
    "GO:0043217",  # myelin maintenance
    "GO:0021675",  # nerve development
    "GO:0008347",  # glial cell migration
    "GO:0048143",  # astrocyte activation
    "GO:0042474",  # middle ear morphogenesis
    # 粒细胞趋化与活化（非巨噬 / 内皮主线）
    "GO:0048245",  # eosinophil chemotaxis
    "GO:0042119",  # neutrophil activation
    "GO:0030593",  # neutrophil chemotaxis
    "GO:0002446",  # neutrophil mediated immunity
    # 骨骼 / 肌肉 / 胚胎附属结构
    "GO:0035988",  # chondrocyte proliferation
    "GO:0030282",  # bone mineralization
    "GO:0061061",  # muscle structure development
    "GO:0014812",  # muscle cell migration
    "GO:0060674",  # placenta blood vessel development
    "GO:0021846",  # cell proliferation in forebrain
    # 其他非目标主题
    "GO:0002064",  # epithelial cell development
    "GO:0035701",  # hematopoietic stem cell migration
    "GO:0002244",  # hematopoietic progenitor cell differentiation
    "GO:0050892",  # intestinal absorption
    "GO:0050901",  # leukocyte tethering or rolling
    "GO:1990000",  # amyloid fibril formation
    "GO:0001945",  # lymph vessel development
    "GO:0061298",  # retina vasculature development in camera-type eye
    # ---- 第二批（同一主题域补充剔除；GO:0150078 已在第一批）----
    "GO:0060252",  # positive regulation of glial cell proliferation
    "GO:0019233",  # sensory perception of pain
    "GO:0048711",  # positive regulation of astrocyte differentiation
    "GO:0043525",  # positive regulation of neuron apoptotic process
    "GO:0050954",  # sensory perception of mechanical stimulus
    "GO:0045665",  # negative regulation of neuron differentiation
    "GO:0014059",  # regulation of dopamine secretion
    "GO:1903975",  # regulation of glial cell migration
    "GO:0045672",  # positive regulation of osteoclast differentiation
    "GO:0046888",  # negative regulation of hormone secretion
    # ---- 第三批（骨骼发育补充剔除）----
    "GO:0030278",  # regulation of ossification
    "GO:0001503",  # ossification
    # ---- 第四批（行为/神经/成骨补充剔除）----
    "GO:0001975",  # response to amphetamine
    "GO:0014048",  # regulation of glutamate secretion
    "GO:0010976",  # positive regulation of neuron projection development
    "GO:0045668",  # negative regulation of osteoblast differentiation
}
_before = list(keep_ids)
keep_ids = [g for g in _before if g not in EXCLUDE_GO]
_excluded = [g for g in _before if g in EXCLUDE_GO]
print(f"[GO:BP] 人工剔除无关条目: {len(_excluded)}  ->  剩余 {len(keep_ids)}")

_ex_term = dict(zip(term_df["GOID"], term_df["TERM"]))
pd.DataFrame({"GO.ID": _excluded,
              "term": [_ex_term.get(g, "") for g in _excluded]}) \
    .to_csv(ENR / "GOBP_excluded_terms.tsv", sep="\t", index=False)

# ---- 5. REVIGO（simRel 语义相似度）去冗余 ----
# topGO 的 elim 只调整父节点 p 值：父节点只要还保有独立基因信号就仍显著，
# 于是父子节点会同时留下。补一步 REVIGO 算法（Bioconductor rrvgo 实现）：
#   calculateSimMatrix -> 两两 simRel 语义相似度（GO-DAG + org.Mm.eg.db 信息量）
#   reduceSimMatrix    -> 按相似度聚类，每簇留一个代表（uniqueness/dispensability 排序）
# 打分取该 GO 项在 8 个节点中的最小 elim p（越小越显著）。
_pv_long = pd.DataFrame([{"node": n, "GO.ID": g, "p_elim": v}
                         for n, d in elim_pv.items() for g, v in d.items()])
# rrvgo 的 score 约定：值越大越优先当簇代表（内部 which.max）。
# 因此必须传 -log10(p)；直接传原始 p 会反向选中簇内最不显著的项。
_go_score = -np.log10(
    _pv_long.groupby("GO.ID")["p_elim"].min().clip(lower=1e-300))

ro.r('suppressPackageStartupMessages(library(rrvgo))')
ro.globalenv["rev_ids"] = ro.StrVector(keep_ids)
ro.globalenv["rev_score"] = ro.FloatVector(_go_score.reindex(keep_ids).to_numpy())
ro.r('names(rev_score) <- rev_ids')
ro.r('''
    rev_sim <- calculateSimMatrix(rev_ids, orgdb = "org.Mm.eg.db",
                                  ont = "BP", method = "Rel")
    rev_red <- reduceSimMatrix(rev_sim, rev_score,
                               threshold = 0.7, orgdb = "org.Mm.eg.db")
''')
with localconverter(ro.default_converter + pandas2ri.converter):
    rev_red = ro.conversion.rpy2py(ro.r['rev_red'])
rev_red["go"] = rev_red["go"].astype(str)
rev_red["parent"] = rev_red["parent"].astype(str)

rev_reps = set(rev_red.loc[rev_red["go"] == rev_red["parent"], "go"])
_n_dropped = len(keep_ids) - rev_red["go"].nunique()
print(f"[GO:BP] REVIGO(simRel, threshold=0.7): {len(keep_ids)} -> {len(rev_reps)} 个代表"
      f"（sim 矩阵 {list(ro.r('dim(rev_sim)'))[0]} 项，{rev_red['parent'].nunique()} 簇）")
if _n_dropped > 0:
    print(f"         注：{_n_dropped} 个 GO 项因无注释/不在 BP 本体中被 rrvgo 略过")

# 输出时把 score 的语义写清楚：rrvgo 的 score 实为传入的 -log10(p)，
# 另附还原后的 p_elim_min，避免与 GOBP_pre_revigo_terms.tsv 里的原始 p 混淆。
_pmin = _pv_long.groupby("GO.ID")["p_elim"].min()
_rev_out = rev_red.rename(columns={"go": "GO.ID", "parent": "rep_GO.ID",
                                   "score": "score_neg_log10_p"}).copy()
_rev_out["p_elim_min"] = _rev_out["GO.ID"].map(_pmin)
_rev_out[["GO.ID", "rep_GO.ID", "cluster", "score_neg_log10_p", "p_elim_min",
          "size", "term", "parentTerm", "termDispensability"]] \
    .to_csv(ENR / "GOBP_revigo_clusters.tsv", sep="\t", index=False)
print("         REVIGO 聚类明细 -> results/enrichment/GOBP_revigo_clusters.tsv")

all_elim_short = {id2short[g] for g in rev_reps if g in id2short}
print(f"[GO:BP] 最终代表（映射回 MSigDB 名称）: {len(all_elim_short)}")

# ---- 6. 通用：基因集包含关系 + Jaccard 合并（子节点优先）----
def dedup_by_gene_overlap(rows_df, genes_map, jaccard_thr=0.85):
    pwids = rows_df["pathway"].tolist()
    n = len(pwids)
    if n == 0: return []
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry: parent[ry] = rx

    sizes = {p: len(genes_map.get(p, set())) for p in pwids}
    nes = dict(zip(rows_df["pathway"], rows_df["max_abs_NES"]))

    for i in range(n):
        for j in range(i + 1, n):
            a, b = pwids[i], pwids[j]
            ga, gb = genes_map.get(a, set()), genes_map.get(b, set())
            if not ga or not gb: continue
            u = len(ga | gb)
            if u > 0 and len(ga & gb) / u > jaccard_thr:
                union(i, j)

    groups = {}
    for i in range(n): groups.setdefault(find(i), []).append(pwids[i])

    reps = []
    for members in groups.values():
        if len(members) == 1:
            reps.append(members[0])
        else:
            reps.append(sorted(members, key=lambda p: (sizes[p], -nes.get(p, 0)))[0])
    return reps

# ---- 7. 构建 Reactome / Hallmark 的基因集 ----
def load_gmt_genes(path, prefix):
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.strip().split("\t")
            if not parts[0].startswith(prefix): continue
            out[parts[0]] = set(parts[2:]) & set(stat.columns)
    return out

reactome_genes = load_gmt_genes(
    ROOT / "data_interim" / "gmt" / "m2.cp.v2026.1.Mm.symbols.gmt", "REACTOME_")
hallmark_genes = load_gmt_genes(
    ROOT / "data_interim" / "gmt" / "mh.all.v2026.1.Mm.symbols.gmt", "HALLMARK_")

# ---- 8. 每个 collection 的显著通路 ----
def sig_rows(coll):
    sub = act[act.collection == coll][["pathway", "max_abs_NES"]].copy()
    if sub.empty: return sub
    return (sub.sort_values("max_abs_NES", ascending=False)
               .drop_duplicates("pathway").reset_index(drop=True))

reactome_sig = sig_rows("reactome")
hallmark_sig = sig_rows("hallmark")

reactome_reps = dedup_by_gene_overlap(reactome_sig, reactome_genes, jaccard_thr=0.85)
hallmark_reps = dedup_by_gene_overlap(hallmark_sig, hallmark_genes, jaccard_thr=0.85)

print(f"[Reactome] 显著 {len(reactome_sig)} -> 去冗余后 {len(reactome_reps)}")
print(f"[Hallmark] 显著 {len(hallmark_sig)} -> 去冗余后 {len(hallmark_reps)}")

# ---- 9. 构建与旧 CL 完全兼容的新 CL ----
all_sig_keys = set(zip(act.collection, act.pathway))
nes_mat = L.pivot_table(index=["collection", "pathway"], columns="node",
                        values="NES", observed=True).reindex(columns=NODE_ORDER)

def build_coll_df(coll, rep_list):
    keys = [k for k in nes_mat.index if k[0] == coll and k in all_sig_keys]
    if not keys:
        return pd.DataFrame(columns=["collection", "pathway", "cluster",
                                     "max_abs_NES", "pathway_short",
                                     "is_representative", "cluster_size"])
    M = nes_mat.loc[keys]
    tmp = pd.DataFrame({
        "collection": [k[0] for k in keys],
        "pathway": [k[1] for k in keys],
        "max_abs_NES": np.abs(M.values).max(axis=1),
    })
    tmp["pathway_short"] = (tmp["pathway"]
                            .str.replace("HALLMARK_", "", regex=False)
                            .str.replace("REACTOME_", "", regex=False)
                            .str.replace("GOBP_", "", regex=False))
    rep_set = set(rep_list)
    tmp["is_representative"] = tmp["pathway"].isin(rep_set)
    tmp["cluster"] = coll + "_" + tmp.index.astype(str)
    tmp["cluster_size"] = 1
    return tmp

go_reps_full = [f"GOBP_{p}" for p in all_elim_short]
go_df = build_coll_df("go_bp", go_reps_full)
react_df = build_coll_df("reactome", reactome_reps)
hall_df = build_coll_df("hallmark", hallmark_reps)

CL = pd.concat([hall_df, react_df, go_df], ignore_index=True)
CL = CL.sort_values(["collection", "is_representative", "max_abs_NES"],
                    ascending=[True, False, False]).reset_index(drop=True)

CL.to_csv(ENR / "redundancy_clusters.tsv", sep="\t", index=False)
CL[CL.is_representative][["collection", "cluster", "cluster_size", "pathway_short"]] \
    .to_csv(ENR / "redundancy_representatives.tsv", sep="\t", index=False)

print("\n--- 最终去冗余汇总 ---")
print(CL.groupby("collection").agg(
    显著通路=("pathway", "nunique"),
    代表数=("is_representative", "sum"),
))


R callback write-console: 
groupGOTerms: 	GOBPTerm, GOMFTerm, GOCCTerm environments built.
  


R callback write-console: 
Building most specific GOs .....
  


R callback write-console: 	( 9764 GO terms found. )
  


R callback write-console: 
Build GO DAG topology ..........
  


R callback write-console: 	( 12748 GO terms and 27997 relations. )
  


...

R callback write-console: preparing gene to GO mapping data...
  


R callback write-console: preparing IC data...
  


R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


[GO:BP] REVIGO(simRel, threshold=0.7): 199 -> 42 个代表（sim 矩阵 199 项，42 簇）
         REVIGO 聚类明细 -> results/enrichment/GOBP_revigo_clusters.tsv
[GO:BP] 最终代表（映射回 MSigDB 名称）: 42


[Reactome] 显著 268 -> 去冗余后 245
[Hallmark] 显著 33 -> 去冗余后 33

--- 最终去冗余汇总 ---
            显著通路  代表数
collection           
go_bp       1386   42
hallmark      33   33
reactome     268  245


In [8]:
summ = (L.groupby(["collection", "node"], observed=True)
          .apply(lambda g: pd.Series({
              "n_tested": len(g),
              "n_sig": int(g.significant.sum()),
              "n_sig_up": int((g.significant & (g.NES > 0)).sum()),
              "n_sig_down": int((g.significant & (g.NES < 0)).sum()),
              "min_padj": float(g.padj.min())}), include_groups=False)
          .reset_index())
summ.to_csv(ENR / "enrichment_per_node_summary.tsv", sep="\t", index=False)
print(summ.pivot(index="node", columns="collection", values="n_sig").reindex(NODE_ORDER))

collection                         go_bp  hallmark  reactome
node                                                        
Heart_and_Aorta__endothelial cell  278.0      17.0     110.0
Heart_and_Aorta__macrophage        261.0      16.0      23.0
Kidney__endothelial cell            26.0      10.0     130.0
Kidney__macrophage                 160.0      11.0      30.0
Liver__endothelial cell            342.0       9.0      27.0
Liver__macrophage                  872.0      20.0      92.0
Lung__endothelial cell             245.0      10.0      27.0
Lung__macrophage                   417.0      14.0      90.0


## 5. 交付物③ Fig3（四面板）

In [9]:
de = pd.read_csv(RES / "de" / "DE_summary.tsv", sep="\t")
de["node"] = de["tissue"] + "__" + de["cell_type"]
de = de.set_index("node").reindex(NODE_ORDER).reset_index()

NLAB = {n: n.replace("Heart_and_Aorta__", "Heart|").replace("__", "|") for n in NODE_ORDER}
C_UP, C_DN = "#c0392b", "#2c6fbb"
plt.rcParams.update({"font.size": 8, "pdf.fonttype": 42, "ps.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---- 构建去冗余后的 act_rep 用于面板 (b) 和 (d) ----
keep_keys = set(zip(CL[CL.is_representative].collection, CL[CL.is_representative].pathway))
act_rep = act[[ (c, p) in keep_keys for c, p in zip(act.collection, act.pathway) ]].copy()

fig = plt.figure(figsize=(14.6, 9.6))
gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.42,
                      left=0.205, right=0.98, top=0.93, bottom=0.10)

# (a) 逐节点 DE 基因数
ax = fig.add_subplot(gs[0, 0])
y = np.arange(len(NODE_ORDER))[::-1]
ax.barh(y, de["n_up"], color=C_UP, height=0.7, label="up in old")
ax.barh(y, -de["n_down"], color=C_DN, height=0.7, label="down in old")
for yy, u, dn in zip(y, de["n_up"], de["n_down"]):
    ax.text(u + 6, yy, str(int(u)), va="center", fontsize=7, color=C_UP)
    ax.text(-dn - 6, yy, str(int(dn)), va="center", ha="right", fontsize=7, color=C_DN)
ax.set_yticks(y); ax.set_yticklabels([NLAB[n] for n in NODE_ORDER], fontsize=7.5)
ax.set_xlabel("number of significant genes (FDR < 0.05)", fontsize=8)
ax.set_title("(a) DE genes per node (old vs young, PyDESeq2 Wald)", fontsize=9.5)
mx = max(de["n_up"].max(), de["n_down"].max()) * 1.22
ax.set_xlim(-mx, mx); ax.axvline(0, color="#888", lw=0.8)
ax.legend(fontsize=7.5, frameon=False, loc="lower right")

# (b) 模块 x 集合（已应用去冗余）
ax = fig.add_subplot(gs[0, 1])
tab = (act_rep.groupby(["collection", "module"], observed=True).size().unstack(fill_value=0)
          .reindex(columns=["Shared", "Partial", "Specific"], fill_value=0)
          .reindex(["hallmark", "reactome", "go_bp"]).fillna(0))
tab.index = ["Hallmark", "Reactome", "GO:BP"]
x = np.arange(len(tab)); w = 0.26
for i, (m, c) in enumerate(zip(["Shared", "Partial", "Specific"],
                               ["#c0392b", "#e8a33d", "#4a6f97"])):
    ax.bar(x + (i - 1) * w, tab[m], w, color=c,
           label=f"{m} ({['\u22653', '=2', '=1'][i]} tissue)")
    for xx, v in zip(x + (i - 1) * w, tab[m]):
        ax.text(xx, v * 1.03, str(int(v)), ha="center", fontsize=7)
ax.set_yscale("log"); ax.set_ylim(0.7, tab.values.max() * 3.2)
ax.set_xticks(x); ax.set_xticklabels(tab.index, fontsize=8)
ax.set_ylabel("n pathway x direction (log)", fontsize=8)
ax.set_title("(b) cross-tissue spread of significant pathways", fontsize=9.5)
ax.legend(fontsize=7, frameon=False, loc="upper left")

# (c) top 共享通路（去冗余）
ax = fig.add_subplot(gs[1, 0])
sh = act[act.module == "Shared"].copy()
keep = set(zip(CL[CL.is_representative].collection, CL[CL.is_representative].pathway))
sh_rep = sh[[(c, p) in keep for c, p in zip(sh.collection, sh.pathway)]]
top = sh_rep.sort_values("max_abs_NES", ascending=False).head(15).iloc[::-1]
cols = [C_UP if d == "up_in_old" else C_DN for d in top.direction]
yy = np.arange(len(top))
ax.barh(yy, top.max_abs_NES, color=cols, height=0.72)
def _short(x, k=33):
    return x if len(x) <= k else x[:k - 1].rstrip("_") + "\u2026"
ax.set_yticks(yy)
ax.set_yticklabels([f"{_short(p)}  [{int(n)}]" for p, n in zip(top.pathway_short, top.n_tissues)],
                   fontsize=6.6)
ax.set_xlabel("max |NES| among active nodes", fontsize=8)
ax.set_title("(c) top shared pathways (>=3 tissues, de-duplicated)   [n] = #tissues", fontsize=9.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=C_UP, label="up in old"), Patch(color=C_DN, label="down in old")],
          fontsize=7, frameon=False, loc="lower right")
ax.set_xlim(0, top.max_abs_NES.max() * 1.18)

# (d) 组织特异（已应用去冗余）
ax = fig.add_subplot(gs[1, 1])
sp = act_rep[act_rep.module == "Specific"]
cnt = (sp.groupby("collection", observed=True).apply(lambda g: g["tissues"].value_counts(),
                                                     include_groups=False).unstack(fill_value=0)
         .reindex(["hallmark", "reactome", "go_bp"]).fillna(0).astype(float)
         .reindex(columns=TISSUES, fill_value=0))
x = np.arange(len(TISSUES)); w = 0.26
for i, (cn, col) in enumerate(zip(["Hallmark", "Reactome", "GO:BP"],
                                  ["#c0392b", "#e8a33d", "#4a6f97"])):
    ax.bar(x + (i - 1) * w, cnt.loc[cnt.index[i]], w, color=col, label=cn)
    for xx, v in zip(x + (i - 1) * w, cnt.loc[cnt.index[i]]):
        if v > 0:
            ax.text(xx, v * 1.04, str(int(v)), ha="center", fontsize=6.8)
ax.set_yscale("log"); ax.set_ylim(0.7, max(cnt.values.max() * 3.5, 3))
ax.set_xticks(x); ax.set_xticklabels(["Heart & Aorta", "Kidney", "Liver", "Lung"], fontsize=8)
ax.set_ylabel("n tissue-specific pathways (log)", fontsize=8)
ax.set_title("(d) tissue-specific pathways (active in exactly 1 tissue)", fontsize=9.5)
ax.legend(fontsize=7, frameon=False)

fig.suptitle("Figure 3 | pseudobulk differential expression and pathway enrichment "
             "(8 nodes = 4 tissues x 2 cell types)", fontsize=11.5, y=0.975)
for ext in ("pdf", "png"):
    fig.savefig(FIG / f"Fig3_DE_pathway_draft.{ext}", dpi=300)
plt.close(fig)
print("saved -> figures/main/Fig3_DE_pathway_draft.pdf|png")

saved -> figures/main/Fig3_DE_pathway_draft.pdf|png


## 6. GO:BP 去冗余结果热图（全部代表，8 节点同时展示）

GO:BP 的去冗余用 **topGO（elim + Fisher）**：逐节点以 BH-FDR < 0.05 的 DE 基因做 GO:BP 富集，
elim 自底向上遍历 GO-DAG，子项显著时把其注释的基因从祖先项扣除后再检验，使「仅继承子项信号」的
祖先项不再显著。跨节点取并集后映射回 MSigDB 名称，再与 GSEA 显著条目取交集，得 **244 条代表**。

本节把这 244 条代表**全部**画出，不做主题筛选、不再二次归并：8 个 tissue × cell 节点并列，
格内按 **NES** 上色，`*` 标出 GSEA BH-FDR < 0.05 的格。行按「显著节点数 → max|NES|」降序排列，
便于把跨节点一致的通路排在上方。


In [10]:
# ---- GO:BP 去冗余（topGO elim + REVIGO）后全部代表：8 节点 NES 热图，* = BH-FDR < 0.05 ----
rep = CL[(CL.collection == "go_bp") & CL.is_representative].copy()
rep["pw"] = rep["pathway"].str.replace("GOBP_", "", regex=False)

G = L[L.collection == "go_bp"].copy()
G["pw"] = G["pathway"].str.replace("GOBP_", "", regex=False)
nes_g = G.pivot(index="pw", columns="node", values="NES").reindex(columns=NODE_ORDER)
padj_g = G.pivot(index="pw", columns="node", values="padj").reindex(columns=NODE_ORDER)

pw = [p for p in rep["pw"] if p in nes_g.index]
order = (pd.DataFrame({"k": padj_g.loc[pw].lt(ALPHA).sum(axis=1),
                       "m": nes_g.loc[pw].abs().max(axis=1)})
         .sort_values(["k", "m"], ascending=False).index.tolist())
Hm, Pm = nes_g.loc[order], padj_g.loc[order]
nR, nC = Hm.shape
print(f"GO:BP 去冗余代表 {nR} 条（全部画出）| 显著格 "
      f"{int(Pm.lt(ALPHA).values.sum())} / {nR * nC}")

# 行高与字号随行数自适应：行少时放大，行多时压缩
row_h = 0.26 if nR <= 60 else 0.13
lab_fs = 7.0 if nR <= 60 else 4.2
star_fs = 9.0 if nR <= 60 else 6.5
fig_h = row_h * nR + 2.2

fig, ax = plt.subplots(figsize=(10.4, fig_h))
cmap = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("white")
im = ax.imshow(Hm.values, aspect="auto", cmap=cmap, vmin=-2.6, vmax=2.6)
for i in range(nR):
    for j in range(nC):
        if Pm.values[i, j] < ALPHA:
            ax.text(j, i, "*", ha="center", va="center", fontsize=star_fs, zorder=5,
                    color=("white" if abs(Hm.values[i, j]) > 1.2 else "black"))
for i in range(nR + 1):
    ax.axhline(i - 0.5, color="white", lw=0.5, zorder=3)
for j in range(nC + 1):
    ax.axvline(j - 0.5, color="white", lw=0.9, zorder=3)

ax.set_yticks(range(nR)); ax.set_yticklabels(order, fontsize=lab_fs)
ax.tick_params(axis="y", length=0, pad=3)
ax.set_xticks(range(nC))


def _xlab(node):
    tissue, ct = node.split("__")
    tissue = "Heart" if tissue == "Heart_and_Aorta" else tissue
    return f"{tissue}|{'endo' if ct == 'endothelial cell' else 'macro'}"


ax.set_xticklabels([_xlab(n) for n in NODE_ORDER], rotation=45, ha="right",
                   fontsize=8, rotation_mode="anchor")
ax.set_xlim(-0.5, nC - 0.5)
cb = fig.colorbar(im, ax=ax, fraction=0.026, pad=0.02, shrink=min(0.35, 2.2 / fig_h))
cb.set_label("NES", fontsize=8); cb.ax.tick_params(labelsize=7)

# 标题用「绝对英寸偏移」定位，图变矮时两行也不会叠在一起
fig.text(0.5, 1 - 0.34 / fig_h,
         f"GO:BP — {nR} representatives after topGO elim + REVIGO (all shown)",
         ha="center", va="top", fontsize=10)
fig.text(0.5, 1 - 0.64 / fig_h,
         f"coloured by NES  |  * = GSEA BH-FDR < {ALPHA}",
         ha="center", va="top", fontsize=9)

fig.subplots_adjust(left=0.30, right=0.90, top=1 - 0.95 / fig_h,
                    bottom=min(0.12, 1.05 / fig_h))
fig.canvas.draw()
_r = fig.canvas.get_renderer()
_wmax = max(t.get_window_extent(_r).width for t in ax.get_yticklabels())
fig.subplots_adjust(left=min(max((_wmax + 16) / fig.bbox.width, 0.25), 0.62))
fig.canvas.draw()

for ext in ("png", "pdf"):
    fig.savefig(FIG / f"Fig3_GOBP_all_representatives.{ext}", dpi=300)
plt.close(fig)
print("saved -> figures/main/Fig3_GOBP_all_representatives.pdf|png")

pd.DataFrame([{"pathway": p,
               "n_sig_nodes": int(padj_g.loc[p].lt(ALPHA).sum()),
               "max_abs_NES": float(nes_g.loc[p].abs().max())}
              for p in order]) \
    .to_csv(ENR / "GOBP_dedup_representatives.tsv", sep="\t", index=False)
print(f"代表清单 -> results/enrichment/GOBP_dedup_representatives.tsv ({nR} 行)")


GO:BP 去冗余代表 42 条（全部画出）| 显著格 81 / 336


saved -> figures/main/Fig3_GOBP_all_representatives.pdf|png
代表清单 -> results/enrichment/GOBP_dedup_representatives.tsv (42 行)
